# Week 5 Lab: Quantization — Toolbox, Measurement, Deployment

**ECBS5200 — Practical Deep Learning Engineering**

Week 3 shipped two fine-tuned models. Week 4 diagnosed where they break.
This week you compress them.

> **Quantization is a toolbox, not a technique.** Each method targets a
> different constraint — training memory, deployment scale, inference
> latency, edge hardware. The trade-off depends on your tool, your model,
> and your hardware. Most claims are true somewhere and false elsewhere.
> Measure per-tier on your setup before you ship.

Today you'll take the encoder and decoder you fine-tuned in Week 3, quantize
them to int8 and int4, and measure four things on each: macro F1, per-tier
Δaccuracy, latency, and peak VRAM. Then you'll measure calibration (ECE)
under compression. You will leave with a six-row table of evidence and a
deployment decision you can defend from it.

**Time budget:** ~80 minutes.

**How to use this notebook:**
- **GUIDED** cells run as-is. Read them.
- **INTERACTIVE** cells require you to fill in values or write answers.
- `___` is a placeholder that will cause a NameError if not filled in.
- Some sections ask you to **PREDICT** before observing — write down your
  prediction first, then reveal.

## Kaggle setup (do this first!)

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → GPU T4
4. **Secrets** → add `HF_TOKEN` (HuggingFace read token)

In [ ]:
import subprocess, sys, os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# HuggingFace config — set BEFORE importing HF libraries
if os.path.exists('/kaggle/working'):
    os.environ['HF_HOME'] = '/kaggle/working/.hf_cache'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

# bitsandbytes is the new dependency this week — the int8 and int4 loading paths
# route through it. accelerate is already installed on Kaggle but we pin here
# because bitsandbytes wants a compatible version.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "bitsandbytes>=0.43", "transformers>=4.53", "datasets", "accelerate",
    "scikit-learn", "matplotlib", "pandas", "tqdm", "peft",
])
print("Packages installed.")

from huggingface_hub import login
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace.")
else:
    print("WARNING: No HF_TOKEN found. May hit rate limits.")

In [ ]:
import gc
import time
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    BitsAndBytesConfig,
)
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# Course utils — try relative path first, clone if missing
REPO_PATH = Path("../..").resolve()
if not (REPO_PATH / "utils" / "data_utils.py").exists():
    REPO_PATH = Path("/tmp/course")
    if not REPO_PATH.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/earino/applied-deep-learning.git", str(REPO_PATH)])
sys.path.insert(0, str(REPO_PATH))

from utils.data_utils import (
    load_course_data, LABEL_LIST, NUM_LABELS, MAX_LENGTH,
)

In [ ]:
# Reproducibility and device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    cc = torch.cuda.get_device_capability(0)
    print(f"  compute capability: {cc[0]}.{cc[1]}")
    print(f"  count: {torch.cuda.device_count()}")
else:
    print("No GPU detected — Act 1 and 2 will fail; rerun on a T4.")

## Load the canonical val set

Same split you've been using all course. 6,430 val examples across 113
classes. The tier definitions (head/mid/tail by training frequency) you
saw in Week 4 return here — alongside the aggregate macro-F1, we'll
break numbers out per tier so you can see whether any pattern hides
inside the aggregate.

In [ ]:
# GUIDED: Data
train_ds, val_ds, _ = load_course_data()
val_texts = list(val_ds["text"])
val_labels = np.array(val_ds["label"])
train_labels = np.array(train_ds["label"])
train_support = Counter(train_labels.tolist())
print(f"Val: {len(val_texts):,} examples, {NUM_LABELS} classes")

In [ ]:
# GUIDED: Tier definitions — same scheme as Week 4
# head = top 20 classes by training frequency
# mid  = next 40
# tail = bottom 53
support_df = pd.DataFrame({
    "class": list(range(NUM_LABELS)),
    "train_n": [train_support.get(c, 0) for c in range(NUM_LABELS)],
})
support_df["val_n"] = [int((val_labels == c).sum()) for c in range(NUM_LABELS)]
support_df = support_df.sort_values("train_n", ascending=False).reset_index(drop=True)
support_df["rank"] = range(1, len(support_df) + 1)
support_df["tier"] = "mid"
support_df.loc[:19, "tier"] = "head"
support_df.loc[60:, "tier"] = "tail"
tier_by_class = dict(zip(support_df["class"], support_df["tier"]))
val_tiers = np.array([tier_by_class[int(c)] for c in val_labels])
print(f"head: {(val_tiers == 'head').sum():5d} val examples across 20 classes")
print(f"mid:  {(val_tiers == 'mid').sum():5d} val examples across 40 classes")
print(f"tail: {(val_tiers == 'tail').sum():5d} val examples across 53 classes")

### Three classes you'll track by name

113 classes is too many to hold in your head. For the rest of the lab
we'll keep an eye on three specific classes — one per tier — so you can
watch what happens to concrete predictions as precision drops.

- **Head (rank 1):** *"Incorrect information on your report"* — by far
  the most common complaint in the dataset (13k+ training examples).
- **Mid (rank 57):** *"Managing, opening, or closing your mobile wallet
  account"* — moderate support (126 training examples, 14 val).
- **Tail (rank 96, n_val=2):** *"Cash advance fee"* — a tail class that
  meets our selection rule of n_val ≥ 2. Because n_val is 2, one flipped
  prediction moves per-class accuracy by 50 points; treat per-class
  dynamics on this row as directional, not significant.

In [ ]:
# GUIDED: Track these three classes by name
NAMED_CLASSES = {
    "head": "Incorrect information on your report",
    "mid":  "Managing, opening, or closing your mobile wallet account",
    "tail": "Cash advance fee",
}

def name_to_id(name):
    """Exact-match class name → class id. Fails loudly if not found."""
    if name not in LABEL_LIST:
        # Try substring fallback (class names can be slightly reworded)
        hits = [i for i, n in enumerate(LABEL_LIST) if name.lower() in n.lower()]
        if not hits:
            hits = [i for i, n in enumerate(LABEL_LIST)
                    if name.split(",")[0].lower() in n.lower()]
        if not hits:
            raise ValueError(f"Class '{name}' not found in LABEL_LIST")
        return hits[0]
    return LABEL_LIST.index(name)

NAMED_IDS = {tier: name_to_id(n) for tier, n in NAMED_CLASSES.items()}
for tier, cls_id in NAMED_IDS.items():
    row = support_df[support_df["class"] == cls_id].iloc[0]
    print(f"  {tier:5s} [{cls_id:3d}] \"{LABEL_LIST[cls_id]}\"  "
          f"(rank {int(row['rank'])}, n_train={int(row['train_n'])}, "
          f"n_val={int(row['val_n'])})")

---
## Part 1: Helper Contract — `measure_quantized_model`

Quantization work has a lot of ceremony — BitsAndBytesConfig flags,
tokenizer padding sides, latency timing with CUDA sync, VRAM measurement
with peak stats reset, bootstrap CIs. We'll provide that as a single
helper you call six times (2 models × 3 precisions). You focus on
interpreting the numbers.
---

In [ ]:
# GUIDED: precision-aware model loading
def load_model_at_precision(model_id, precision, num_labels, fix_pad_token_id=None):
    """Load a model at fp16 / int8 / int4 via bitsandbytes.

    device_map={"": 0} pins everything on cuda:0 (important on Kaggle's dual-T4
    instance — "auto" can shard across both GPUs and trip cross-device errors).
    """
    kwargs = {"num_labels": num_labels, "device_map": {"": 0}}
    if precision == "fp16":
        kwargs["dtype"] = torch.float16
    elif precision == "int8":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif precision == "int4":
        # NF4 + double quantization + fp16 compute dtype.
        # This is the QLoRA training recipe, reused at inference time.
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        raise ValueError(f"Unknown precision: {precision}")
    model = AutoModelForSequenceClassification.from_pretrained(model_id, **kwargs)
    if fix_pad_token_id is not None and model.config.pad_token_id is None:
        model.config.pad_token_id = fix_pad_token_id
    return model

In [ ]:
# GUIDED: batched inference — returns logits/probs/preds, all fp32 on CPU
def run_inference(model, tokenizer, texts, batch_size=32, max_length=MAX_LENGTH):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=max_length).to(model.device)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = model(**enc)
            all_logits.append(out.logits.float().cpu().numpy())
    return np.concatenate(all_logits, axis=0)

In [ ]:
# GUIDED: latency — 3 warmup + 5 timed, median, CUDA-synced
# T4 instances on Kaggle are shared and noisy. A single timed run can swing 20%+.
# Median of 5 after 3 warmups gives a stable number. DON'T use mean — one cold
# outlier dominates it.
def measure_latency(model, tokenizer, sample_texts, batch_size=32,
                    n_warmup=3, n_timed=5, max_length=MAX_LENGTH):
    model.eval()
    batches = []
    for i in range(n_warmup + n_timed):
        start = (i * batch_size) % max(1, len(sample_texts) - batch_size)
        batch = sample_texts[start:start + batch_size]
        if len(batch) < batch_size:
            batch = batch + sample_texts[:batch_size - len(batch)]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_length).to(model.device)
        batches.append(enc)
    with torch.no_grad():
        for i in range(n_warmup):
            with torch.amp.autocast("cuda", dtype=torch.float16):
                _ = model(**batches[i])
    torch.cuda.synchronize()
    times_ms = []
    with torch.no_grad():
        for i in range(n_warmup, n_warmup + n_timed):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.amp.autocast("cuda", dtype=torch.float16):
                _ = model(**batches[i])
            torch.cuda.synchronize()
            times_ms.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times_ms) / batch_size)  # ms per example

In [ ]:
# GUIDED: Top-level helper — one call gets you all the numbers for one config
def measure_quantized_model(model_id, precision, tokenizer_cfg,
                             texts, labels, tiers, batch_size=32):
    """Load a model at the given precision, measure everything, return a dict.

    Metrics returned:
      - accuracy, macro_f1
      - latency_ms_per_ex (median, batched)
      - peak_vram_gb
      - probs, preds (arrays — saved so you can compute Δacc and ECE later)
      - batch_size_used (may drop on OOM)

    OOM-fallback: walks a retry ladder (passed batch_size → 16 → 8 → 4) on
    OOM, halving each step. Fails hard only after the smallest size OOMs.
    """
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_cfg["id"])
    if tokenizer_cfg["fix_pad_token"] and tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = tokenizer_cfg["padding_side"]
    fix_pad_id = tokenizer.pad_token_id if tokenizer_cfg["fix_pad_token"] else None

    model = load_model_at_precision(model_id, precision, NUM_LABELS, fix_pad_id)

    bs_ladder, b = [], batch_size
    while b >= 4:
        bs_ladder.append(b)
        b //= 2
    last_err, logits = None, None
    for bs in bs_ladder:
        try:
            logits = run_inference(model, tokenizer, texts, batch_size=bs)
            break
        except torch.cuda.OutOfMemoryError as err:
            print(f"    OOM at bs={bs} → trying smaller batch")
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            last_err = err
    if logits is None:
        raise RuntimeError(f"Inference OOMed at every batch size in {bs_ladder}") from last_err

    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
    preds = probs.argmax(-1)
    latency = measure_latency(model, tokenizer, texts[:256], batch_size=bs)
    vram_gb = torch.cuda.max_memory_allocated() / 1e9

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "precision": precision,
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "latency_ms_per_ex": float(latency),
        "peak_vram_gb": float(vram_gb),
        "batch_size_used": bs,
        "logits": logits,
        "probs": probs,
        "preds": preds,
    }

In [ ]:
# GUIDED: Model configs — the two Week 3 checkpoints, merged, on HF Hub
MODELS = {
    "encoder": {
        "id": "earino/ecbs5200-week3-encoder-lora",
        "padding_side": "right",
        "fix_pad_token": False,
    },
    "decoder": {
        "id": "earino/ecbs5200-week3-decoder-lora",
        "padding_side": "left",
        "fix_pad_token": True,
    },
}

# Results bank — will accumulate 6 entries (2 models × 3 precisions)
results = {"encoder": {}, "decoder": {}}

---
## Act 1: fp16 baseline + int8 (~30 min)

First the baselines. Load each model at fp16 (the training/inference
default on T4), measure everything. These are your reference points —
every int8 and int4 number in the rest of the lab is "vs fp16 on this
model."

Then int8. Before you run it, **make a prediction**.
---

In [ ]:
# GUIDED: fp16 encoder
print("ENCODER @ fp16 ...")
results["encoder"]["fp16"] = measure_quantized_model(
    MODELS["encoder"]["id"], "fp16", MODELS["encoder"],
    val_texts, val_labels, val_tiers,
)
r = results["encoder"]["fp16"]
print(f"  acc {r['accuracy']:.4f}  macro_f1 {r['macro_f1']:.4f}  "
      f"lat {r['latency_ms_per_ex']:.2f} ms/ex  VRAM {r['peak_vram_gb']:.2f} GB")

In [ ]:
# GUIDED: fp16 decoder
print("DECODER @ fp16 ...")
results["decoder"]["fp16"] = measure_quantized_model(
    MODELS["decoder"]["id"], "fp16", MODELS["decoder"],
    val_texts, val_labels, val_tiers,
)
r = results["decoder"]["fp16"]
print(f"  acc {r['accuracy']:.4f}  macro_f1 {r['macro_f1']:.4f}  "
      f"lat {r['latency_ms_per_ex']:.2f} ms/ex  VRAM {r['peak_vram_gb']:.2f} GB")

### INTERACTIVE: Predict before you measure int8

Quantization gets pitched with three promises: smaller weights, lower
peak VRAM, faster inference. Only the first is structural: int8 weights
take 1 byte vs fp16's 2, int4 takes half a byte. Weight storage *must*
drop. **Peak VRAM is a different story** — it includes quantization state,
outlier paths, scratch buffers, and kernel overhead, which can push the
measured number up rather than down. Inference speed depends on the
kernel, the hardware, and the model size.

For each of the three, write down your prediction. For int8 on a T4,
vs fp16 baseline:

1. **Peak VRAM**: will it go DOWN, stay the same, or go UP?
2. **Latency**: will ms-per-example go DOWN, stay the same, or go UP?
3. **Macro F1**: will it go DOWN, stay the same, or go UP?

YOUR PREDICTIONS:

1. VRAM:
2. Latency:
3. Macro F1:


In [ ]:
# INTERACTIVE: Measure int8 — encoder
# Call measure_quantized_model with precision="int8" for the encoder.
# The call pattern matches the fp16 cells above.

print("ENCODER @ int8 ...")
results["encoder"]["int8"] = ___
r = results["encoder"]["int8"]
print(f"  acc {r['accuracy']:.4f}  macro_f1 {r['macro_f1']:.4f}  "
      f"lat {r['latency_ms_per_ex']:.2f} ms/ex  VRAM {r['peak_vram_gb']:.2f} GB")

In [ ]:
# INTERACTIVE: Measure int8 — decoder
print("DECODER @ int8 ...")
results["decoder"]["int8"] = ___
r = results["decoder"]["int8"]
print(f"  acc {r['accuracy']:.4f}  macro_f1 {r['macro_f1']:.4f}  "
      f"lat {r['latency_ms_per_ex']:.2f} ms/ex  VRAM {r['peak_vram_gb']:.2f} GB")

### Compare your numbers to your predictions

Look at your four numbers above (encoder int8 and decoder int8) against
your three predictions. For each of the three promises — macro F1, VRAM,
latency — was your prediction right or wrong? Don't skip this; the memo
rubric rewards specific numbers over folk wisdom.

<details>
<summary><b>After you've reconciled your predictions, click for the mechanism</b></summary>

Whatever direction your measurements went, Dettmers et al. (2022) — the
*LLM.int8()* paper itself — documents the sub-6.7B regime explicitly:
"The quantization overhead can slow inference for models with less than
6.7B parameters, as compared to a FP16 baseline." Our encoder is 149M
and our decoder is 494M. We sit firmly below that threshold.

The LLM.int8 algorithm decomposes each matrix multiplication into an
INT8 fast path and an FP16 outlier path, then recomposes the output.
At large scale the FP16 fraction is tiny and INT8 wins. At small scale
the decomposition overhead dominates the INT8 savings — and the
quantization state (scaling factors, zero points, outlier-branch
allocations, attention scratch) occupies its own memory. What your
specific numbers show is a direct read on which of the three
quantization promises holds for your model at this scale on this tool.

**Production stack — the vocabulary you'll see at work:**

- **bitsandbytes**: the library we just used. Training-memory tool
  (powers QLoRA, which you used in Week 3). Fine for experimentation
  and on-ramp; not the inference tool for small models on Turing/Ampere.
- **AWQ** (Lin et al. 2024, MLSys Best Paper): weight-only int4 with
  activation-aware salient-weight protection. Major HF model families
  ship pre-quantized AWQ checkpoints. **This is what you'd reach for
  at work for int4 inference.**
- **GPTQ** (Frantar 2023): also weight-only int4, the precursor to AWQ.
  Still encountered in the wild; AutoGPTQ wrapper was archived April 2025,
  so GPTQ is moving from frontier to baseline.
- **FP8** (Kurtic et al. 2024, "BF16 or Death"): effectively lossless
  on Llama-3.1 via vLLM. Requires Hopper (H100) or newer.
- **SmoothQuant** (Xiao 2023): W8A8 path — quantizes activations too,
  not just weights. Still the mainstream W8A8 option on Turing hardware
  where FP8 isn't available.
- **TRT-LLM / vLLM / SGLang**: the inference servers that ship these
  kernels to production.

Benchmarking numbers that make this concrete (Qwen2.5-32B via vLLM, 2026):
AWQ + Marlin 741 tok/s, FP16 461 tok/s, **bitsandbytes 168 tok/s**.
AWQ is about 4× faster than bitsandbytes on the same hardware. On real
production tooling, int4 IS the speed win. With bitsandbytes on T4, it
isn't.

The week's thesis stands: *quantization is a toolbox, not a technique.*
Today we used the on-ramp tool. You now know what to reach for at work
when you want the latency win that bitsandbytes didn't deliver.

</details>

---
## Act 2: int4 + Pareto frontier (~25 min)

int4 halves the weight bytes again vs int8 — 0.5 bytes per parameter.
For the decoder (494M params) that's ~0.25 GB of raw weights vs ~1 GB
at fp16. Whether that structural byte-count difference shows up in
peak VRAM, and how accuracy and latency move, is for you to measure.
---

In [ ]:
# INTERACTIVE: Measure int4 — encoder
# Same pattern as int8; precision="int4".

print("ENCODER @ int4 ...")
results["encoder"]["int4"] = ___
r = results["encoder"]["int4"]
print(f"  acc {r['accuracy']:.4f}  macro_f1 {r['macro_f1']:.4f}  "
      f"lat {r['latency_ms_per_ex']:.2f} ms/ex  VRAM {r['peak_vram_gb']:.2f} GB")

In [ ]:
# INTERACTIVE: Measure int4 — decoder
print("DECODER @ int4 ...")
results["decoder"]["int4"] = ___
r = results["decoder"]["int4"]
print(f"  acc {r['accuracy']:.4f}  macro_f1 {r['macro_f1']:.4f}  "
      f"lat {r['latency_ms_per_ex']:.2f} ms/ex  VRAM {r['peak_vram_gb']:.2f} GB")

### The summary table

Assemble the six measurements into one DataFrame. This is the table that
carries you through the deployment discussion at the end of the lab and
into your memo. Save it — you'll reference it for the next hour.

In [ ]:
# GUIDED: Summary table assembly
rows = []
for model_name, model_results in results.items():
    for precision, r in model_results.items():
        rows.append({
            "model": model_name,
            "precision": precision,
            "accuracy": round(r["accuracy"], 4),
            "macro_f1": round(r["macro_f1"], 4),
            "latency_ms": round(r["latency_ms_per_ex"], 2),
            "vram_gb": round(r["peak_vram_gb"], 2),
            "batch_size": r["batch_size_used"],
        })
summary_df = pd.DataFrame(rows).sort_values(["model", "precision"]).reset_index(drop=True)
print(summary_df.to_string(index=False))

### Pareto frontier — quality vs latency, quality vs memory

A single number (macro F1) is the wrong way to compare deployment
options. A deployment has at least three axes — accuracy, latency, VRAM
— and different constraints pick different winners.

Plot the six points on two axes. "Pareto-dominant" means: no other point
is strictly better on ALL axes. On a quality-vs-latency plot, the top-left
points (high F1, low latency) dominate. On a quality-vs-VRAM plot, top-left
points (high F1, low VRAM) dominate. A point dominated by another on both
plots is one you'd never ship.

In [ ]:
# GUIDED: Pareto plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = {"encoder": "#2166AC", "decoder": "#B2182B"}
markers = {"fp16": "o", "int8": "s", "int4": "^"}

# (a) F1 vs latency
ax = axes[0]
for model_name, model_results in results.items():
    for precision, r in model_results.items():
        ax.scatter(r["latency_ms_per_ex"], r["macro_f1"],
                   c=colors[model_name], marker=markers[precision],
                   s=220, edgecolor="white", linewidth=1.5,
                   label=f"{model_name} {precision}")
        ax.annotate(f"  {model_name[:3]} {precision}",
                    (r["latency_ms_per_ex"], r["macro_f1"]),
                    fontsize=8, alpha=0.7)
ax.set_xlabel("Latency (ms/example, batched bs=32)")
ax.set_ylabel("Macro F1")
ax.set_title("Quality vs latency — top-left dominates")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="best")

# (b) F1 vs VRAM
ax = axes[1]
for model_name, model_results in results.items():
    for precision, r in model_results.items():
        ax.scatter(r["peak_vram_gb"], r["macro_f1"],
                   c=colors[model_name], marker=markers[precision],
                   s=220, edgecolor="white", linewidth=1.5,
                   label=f"{model_name} {precision}")
        ax.annotate(f"  {model_name[:3]} {precision}",
                    (r["peak_vram_gb"], r["macro_f1"]),
                    fontsize=8, alpha=0.7)
ax.set_xlabel("Peak VRAM (GB)")
ax.set_ylabel("Macro F1")
ax.set_title("Quality vs memory — top-left dominates")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="best")

plt.tight_layout()
plt.show()

### INTERACTIVE: Read the Pareto

"Pareto-dominated" means at least one other point beats it on every
axis you care about. Look at both plots and answer:

1. Walk through all six points on each plot. Are there any points you
   would not ship under any latency/VRAM-constrained workload? Name
   them or say none, and justify from the geometry.

2. On quality vs latency, which point would you pick if the ONLY
   constraint is "fastest ms/ex that exceeds macro F1 0.20"?

3. On quality vs VRAM, which point would you pick if the constraint
   is "lowest VRAM that exceeds macro F1 0.20"?

4. Is there a deployment constraint where you'd pick int8 over fp16?
   Name one, or say none.

YOUR ANSWERS:

1.

2.

3.

4.


### Per-tier Δacc under int4

Macro F1 is one aggregate over 113 classes. Per-tier Δaccuracy with
bootstrap CIs lets you see whether any per-tier movement hides inside
that aggregate. This is the core metric for Memo Section 2.

In [ ]:
# GUIDED: Bootstrap for per-tier Δacc — example-level, 1000 resamples, seed=42
def bootstrap_tier_deltas(preds_ref, preds_test, labels, tiers, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    out = {}
    for tier in ["head", "mid", "tail"]:
        idxs = np.where(tiers == tier)[0]
        n = len(idxs)
        deltas = []
        for _ in range(n_boot):
            bi = rng.choice(idxs, n, replace=True)
            acc_ref = (preds_ref[bi] == labels[bi]).mean()
            acc_test = (preds_test[bi] == labels[bi]).mean()
            deltas.append(acc_test - acc_ref)
        deltas = np.array(deltas)
        out[tier] = {
            "n": int(n),
            "mean": float(deltas.mean()),
            "ci_low": float(np.percentile(deltas, 2.5)),
            "ci_high": float(np.percentile(deltas, 97.5)),
        }
    return out

In [ ]:
# INTERACTIVE: Compute tier Δacc at int4 for both models, vs fp16 baseline
# Hint: bootstrap_tier_deltas(preds_fp16, preds_int4, val_labels, val_tiers)

tier_deltas = {}
for model_name in ["encoder", "decoder"]:
    preds_fp16 = results[model_name]["fp16"]["preds"]
    preds_int4 = results[model_name]["int4"]["preds"]
    tier_deltas[model_name] = bootstrap_tier_deltas(___, ___, val_labels, val_tiers)

In [ ]:
# GUIDED: Display
print(f"{'model':<10}{'tier':<8}{'n':>6}{'Δacc':>10}{'95% CI':>22}")
print("-" * 56)
for model_name, td in tier_deltas.items():
    for tier in ["head", "mid", "tail"]:
        t = td[tier]
        ci = f"[{t['ci_low']:+.4f}, {t['ci_high']:+.4f}]"
        excl = "  *" if (t["ci_low"] > 0 or t["ci_high"] < 0) else ""
        print(f"{model_name:<10}{tier:<8}{t['n']:>6}{t['mean']:+10.4f}{ci:>22}{excl}")
print("\n  * = 95% CI excludes zero (internal consistency of this sample; see note below)")

### INTERACTIVE: What does the tier table reveal?

The macro-F1 numbers in your earlier summary table collapsed to one
number per model per precision. The per-tier breakdown shows what went
into that aggregate. Read the table and write down:

1. Describe the tier pattern for the **encoder** at int4 in one sentence.
   (Uniform? Concentrated? Any tier unchanged? Any tier improve?)
2. Same for the **decoder** at int4.
3. Are the two patterns similar, or different?
4. Which tier has the widest confidence interval? What does that width
   tell you about how much to trust the point estimate?

YOUR ANSWERS:

1. Encoder:
2. Decoder:
3. Pattern similarity:
4. Widest CI & interpretation:

A note on what the bootstrap CI does and does not measure.

The tail tier has 210 val examples across 53 classes. A 2-3 pp Δacc
on that tier corresponds to roughly 4-6 flipped predictions. A bootstrap
CI tells you how those specific 4-6 flips would vary under resampling
*this* val set — it does NOT tell you the same effect would appear
with a different stratified split.

Concretely: a narrow bootstrap CI on a tier with few flips is
*directional*, not proof. Two criteria matter for claiming an effect
is real: (a) the CI structure AND (b) whether enough predictions
moved for the result to plausibly survive a reshuffle. A CI that
excludes zero on only ~5 flipped predictions is a "worth investigating
further" finding, not a "ship this to the memo as settled" finding.

Whichever direction your tail rows point, factor the flip count into
your claim strength. The memo rubric rewards this reasoning explicitly.

**This is the content of Memo Section 2.** Pull these numbers directly.

---
## Act 3: Calibration under quantization (~15 min)

From Week 4 you know one fine-tuned model can be 57% accurate and well
calibrated (ECE 0.06), while another 57%-accurate model can be
catastrophically overconfident (ECE 0.10+). Compression may or may not
preserve that property. Your deployment may gate on confidence scores —
if so, calibration under quantization is load-bearing.
---

In [ ]:
# GUIDED: ECE + temperature-scaling — same implementation as Week 4
def compute_ece(probs, labels, n_bins=15):
    confidences = probs.max(axis=-1)
    preds = probs.argmax(axis=-1)
    accuracies = (preds == labels).astype(float)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(labels)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() > 0:
            bin_acc = accuracies[mask].mean()
            bin_conf = confidences[mask].mean()
            ece += (mask.sum() / n) * abs(bin_acc - bin_conf)
    return float(ece)

def temperature_scale(logits, labels, max_iter=50, tol=1e-4):
    """Fit scalar T > 0 minimizing NLL via LBFGS. Returns T.

    Optimizes log_T (T = exp(log_T)) so the constraint T > 0 is structural.
    Warns if the optimizer terminated with a non-trivial gradient (this is
    a post-hoc convergence read on the closure's last grad — LBFGS does
    not always reach the optimum on ill-behaved logits).
    """
    logits_t = torch.from_numpy(logits).float()
    labels_t = torch.from_numpy(labels).long()
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.01, max_iter=max_iter)
    def closure():
        opt.zero_grad()
        T = torch.exp(log_T)
        loss = F.cross_entropy(logits_t / T, labels_t)
        loss.backward()
        return loss
    opt.step(closure)
    final_grad = abs(log_T.grad.item()) if log_T.grad is not None else float("inf")
    if final_grad > tol:
        print(f"    [warn] temperature_scale: |grad|={final_grad:.2e} > {tol:.0e} after {max_iter} LBFGS steps; T may be sub-optimal")
    return float(torch.exp(log_T.detach()).item())

In [ ]:
# INTERACTIVE: Build the 2×3 ECE table — pre and post temperature scaling
# For each (model, precision), compute:
#   - ece_pre = ECE of softmax(logits)
#   - T = fitted scale
#   - ece_post = ECE of softmax(logits / T)

ece_rows = []
for model_name in ["encoder", "decoder"]:
    for precision in ["fp16", "int8", "int4"]:
        r = results[model_name][precision]
        ece_pre = compute_ece(r["probs"], val_labels)
        T = temperature_scale(r["logits"], val_labels)
        # Students fill: probs after T-scaling
        probs_scaled = ___
        ece_post = compute_ece(probs_scaled, val_labels)
        ece_rows.append({
            "model": model_name, "precision": precision,
            "ece_pre": round(ece_pre, 4), "T": round(T, 3),
            "ece_post": round(ece_post, 4),
        })
ece_df = pd.DataFrame(ece_rows)
print(ece_df.to_string(index=False))

### INTERACTIVE: Read the calibration table

1. **Pre-scaling:** describe how ECE moves across precisions (fp16 →
   int8 → int4) for each model. Monotone in either direction? Flat?
   Different pattern on the two models?

2. **Post-scaling:** compare ECE-post at each precision to ECE-post at
   fp16 for the same model. How close does each quantized config get?

3. Write one sentence per model capturing what you'd tell a deployment
   engineer about calibration under compression on this setup. This
   sentence feeds directly into Memo Section 3 — make it specific with
   numbers you just measured.

YOUR INTERPRETATION:

Encoder pattern (Q1):

Decoder pattern (Q1):

Post-scaling recovery, encoder (Q2):

Post-scaling recovery, decoder (Q2):

Deployment sentence, encoder (Q3):

Deployment sentence, decoder (Q3):


---
## Wrap-up (~10 min): Which tool for which constraint?

Pull everything together. You have six-point measurements, a tier-level
breakdown, and a calibration picture.
---

### INTERACTIVE: Hand-raise poll + one-minute decision

Given **everything you just measured**, and imagining this constraint:

> "Single T4 GPU. Must sustain 100 requests/second, batched. Macro F1
> floor 0.20. Post-scaling ECE ceiling 0.05."

Which of the six configurations would you ship, and WHY? You don't need
to be right. You need to name the numbers that forced your choice.

If no configuration meets all four constraints, name which you'd relax
and why.

YOUR RECOMMENDATION:


YOUR JUSTIFICATION (one sentence per constraint):


### The toolbox, named

Today you used **bitsandbytes** — the training and on-ramp tool. It's
honest about what it is: an experimentation path with identical
accuracy-preserving arithmetic to production int8 paths, but with
software-emulated mixed-precision decomposition that pays overhead on
small models.

In 2026 production, you'd reach for different tools:

| Constraint | Tool | Where it lives |
|---|---|---|
| Training memory (QLoRA) | bitsandbytes NF4 | HuggingFace, PEFT |
| Production int4 inference | AWQ | vLLM, TensorRT-LLM, SGLang |
| Modern-hardware inference | FP8 | TransformerEngine on H100/Blackwell |
| CPU / edge | GGUF | llama.cpp |

You haven't used AWQ. You have a reading list entry on it. What you built
today is the measurement discipline that lets you compare any of these
tools on your own data, your own hardware, your own constraints.

Next week: **distillation**. Another tool in the toolbox — different
constraint (shrink the model family itself, not compress weights within
a family), same measurement discipline.

---
## Save results for homework
---

In [ ]:
# GUIDED: Save everything needed for the homework extension
OUT_DIR = Path(".")

# The summary table
summary_df.to_csv(OUT_DIR / "week5_lab_summary.csv", index=False)

# The ECE table
ece_df.to_csv(OUT_DIR / "week5_lab_ece.csv", index=False)

# Tier deltas
with open(OUT_DIR / "week5_lab_tier_deltas.json", "w") as f:
    json.dump(tier_deltas, f, indent=2)

# Logits + preds for homework — 6 arrays, one per config
np.savez_compressed(
    OUT_DIR / "week5_lab_predictions.npz",
    val_labels=val_labels,
    val_tiers=val_tiers,
    encoder_fp16_logits=results["encoder"]["fp16"]["logits"],
    encoder_int8_logits=results["encoder"]["int8"]["logits"],
    encoder_int4_logits=results["encoder"]["int4"]["logits"],
    decoder_fp16_logits=results["decoder"]["fp16"]["logits"],
    decoder_int8_logits=results["decoder"]["int8"]["logits"],
    decoder_int4_logits=results["decoder"]["int4"]["logits"],
)

print("Saved:")
print("  week5_lab_summary.csv       — 6 rows, one per (model, precision)")
print("  week5_lab_ece.csv           — 6 rows, ECE pre/post + T")
print("  week5_lab_tier_deltas.json  — bootstrap per-tier Δacc at int4")
print("  week5_lab_predictions.npz   — logits for all 6 configs")
print("\nDownload these files — the homework reads them.")

### What's next (homework)

In the homework you'll:

1. **Per-tier Δacc at int8 as well as int4** — you've done it for int4
   in the lab; extend to int8 and compare the patterns.
2. **Temperature scaling on split val** — fit T on half of val, evaluate
   ECE reduction on the other half. Does it generalize, or is the fit
   just memorizing?
3. **Named-class trajectories** — track the three named classes (head,
   mid, tail) across fp16 / int8 / int4 for both models. Plot the
   trajectory.
4. **Deployment memo (100 points)** — 5 sections. Rubric is in
   `assessments/week5_memo_rubric.md`. The Excellent-band descriptors
   tell you what mastery of each prompt looks like.